# Playground Series S6E8: Predicting Smartphone Addiction

**notebookタイトル**: What actually moved the score on S6E8 — and what didn't
**原著者**: tomasa2
**元notebookへのリンク**: https://www.kaggle.com/code/tomasa2/s6e8-what-moved-the-score-and-what-didn-t
**スコア**: Public LB 0.96985（本文中の値。Codeタブ一覧では0.96977と表示、ごく僅かな差は再実行や集計タイミングの違いによるものと思われる）

**手法の概要**: 20以上のアイデアを実際に検証し、「何が効いて何が効かなかったか」を定量的に示すEDA＋実験ノートブック。最終的に最も効いたのは、連続値の特徴量も含めて**全カラムを高カーディナリティなカテゴリとしてターゲットエンコーディングする**という手法で、特徴量エンジニアリング・チューニング・アンサンブル・モデル選択を全部合わせたよりも効果が大きかったと報告している。

> これは学習目的の解説付き写しです。元のコードセルはほぼそのまま保持していますが、**Kaggle上のレンダリング済みページからテキストとして抽出したためインデント（字下げ）情報が失われており**、このノートブックのインデントはコード内容の論理構造から筆者（本ルーティンを実行したAIエージェント）が再構成したものです。トークン・処理内容自体は元notebookと同一になるよう努めていますが、フォーマットの完全な忠実再現は保証できません。また実行はしていません（出力セルは元notebookに表示されていた数値をそのまま引用しています）。
>
> なお著者はこのノートブックについて「Built with AI（Claude Codeと協働）」と明記しており、コンペ選定・playbook提供・環境構築・提出判断は著者本人、EDA・実験実装・結果分析はAIエージェントが担当したと述べている。また「一番効いたアイデア（ターゲットエンコーディング）自体はOMID BAGHCHEH SARAEI氏の公開notebookに由来し、著者自身の独自発見ではない」とも率直に述べられている点は、成果を誇張しない誠実な書き方として参考になる。

## 評価指標について

- **タスク**: スマートフォン依存（addicted_label）の二値分類。
- **評価指標**: ROC AUC（受信者操作特性曲線下面積）。
- **特性と指標選定の考察**: 目的変数の陽性率は約0.71と偏りがあり、単純な正解率ではモデルの識別力を正しく評価できないため、閾値に依存しないROC AUCが採用されていると考えられる。ただし本notebookが指摘する通り、target（addicted_label）はスクリーンタイムの多い層でほぼ1.0に張り付く（飽和する）ため、確率のままでは分解能を失いやすい。
- **選んだ手法がどう指標を最適化しているか**: 5-foldクロスバリデーションの再分割誤差（noise floor）を測定してから改善判断を行う、target encodingにより高カーディナリティな数値特徴の非線形な関係を直接学習させる、スタッキングではロジット（対数オッズ）スケールで合成することで確率が飽和する領域でも分解能を保つ、といった工夫がAUCの最適化に直結している。


### セル1: セットアップ（What / Why）

**何をしているか**: 必要なライブラリ（xgboost, lightgbm, catboost, scikit-learn）を読み込み、train/testのCSVパスを複数候補から自動検出する `find_first` ヘルパーを定義。GPUが使えるか試験的に小さなXGBoostモデルを学習して判定する。目的変数 `addicted_label` とカテゴリ列・数値列のリストを定義し、データを読み込む。

**なぜそうするのか**: Kaggle環境ではデータセットのマウントパスが実行環境によって微妙に異なることがあるため、複数パスを順に試す防御的な実装にしている。GPU可否を実際に小さく学習させて確認するのは、`device="cuda"`と書いても実際に使えるとは限らないため（環境依存のエラーを避ける実務的な工夫）。


In [ ]:
import os, glob, time, warnings
import numpy as np, pandas as pd
import xgboost as xgb, lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, r2_score
warnings.filterwarnings("ignore")

def find_first(*pats):
    for p in pats:
        h = sorted(glob.glob(p, recursive=True))
        if h: return h[0]

train_csv = find_first("/kaggle/input/competitions/playground-series-s6e8/train.csv",
                        "/kaggle/input/playground-series-s6e8/train.csv",
                        "/kaggle/input/*/train.csv", "/kaggle/input/**/train.csv",
                        "playground-series-s6e8/train.csv",
                        "../playground-series-s6e8/train.csv")
assert train_csv, "attach the playground-series-s6e8 competition via '+ Add Input'"
DATA = os.path.dirname(train_csv)
ORIG_CSV = find_first("/kaggle/input/**/Smartphone_Usage_And_Addiction*.csv",
                       "../original/**/Smartphone_Usage_And_Addiction*.csv")

DEVICE = "cuda" if os.path.exists("/proc/driver/nvidia") or os.environ.get("CUDA_PATH") else "cpu"
try:
    xgb.XGBClassifier(device="cuda", tree_method="hist", n_estimators=1).fit(
        np.zeros((8, 2)), [0, 1] * 4); DEVICE = "cuda"
except Exception:
    DEVICE = "cpu"

TARGET = "addicted_label"
CATS = ["gender", "stress_level", "academic_work_impact"]
NUMS = ["age", "daily_screen_time_hours", "social_media_hours", "gaming_hours",
        "work_study_hours", "sleep_hours", "notifications_per_day",
        "app_opens_per_day", "weekend_screen_time"]

train = pd.read_csv(f"{DATA}/train.csv"); test = pd.read_csv(f"{DATA}/test.csv")
y = train[TARGET].values
print(f"train {train.shape} test {test.shape} positive rate {y.mean():.4f} device {DEVICE}")


### セル2: ノイズフロア（測定誤差の下限）を先に測る（What / Why）

**何をしているか**: 1回の5-foldクロスバリデーションの中でのfold間スコアのばらつき（per-fold range）を計算した後、分割の乱数シード（partition seed）を変えて複数回クロスバリデーションを繰り返し、その平均値の標準偏差を「ノイズフロア（noise floor）」として算出する。

**なぜそうするのか**: 1回のfold分割内でのばらつきは「別の分割をしたらどれくらいスコアが動くか」を大きく過大評価してしまう。ここでは分割シードを変えて3回クロスバリデーションを繰り返し、その平均スコアのばらつきを見ることで、真の「測定誤差の下限」を求めている。この後に出てくる「+0.001の改善」のような主張が、本当にノイズフロアを超えているのか判断する基準として使われる（実際、ノイズフロアはper-foldレンジの約40分の1しかない、というのが本notebookの重要な発見の一つ）。


In [ ]:
XGB_BASE = dict(n_estimators=4000, learning_rate=0.05, max_depth=6, subsample=0.8,
                colsample_bytree=0.8, min_child_weight=20, tree_method="hist",
                device=DEVICE, enable_categorical=True, eval_metric="auc",
                early_stopping_rounds=100)
mk = lambda seed, **o: xgb.XGBClassifier(**{**XGB_BASE, "random_state": seed, **o})

def cv_once(X, y, seed=42, model_fn=None):
    oof = np.zeros(len(X)); folds = []
    for itr, iva in StratifiedKFold(5, shuffle=True, random_state=seed).split(X, y):
        m = (model_fn or mk)(seed)
        m.fit(X.iloc[itr], y[itr], eval_set=[(X.iloc[iva], y[iva])], verbose=False)
        oof[iva] = m.predict_proba(X.iloc[iva])[:, 1]
        folds.append(roc_auc_score(y[iva], oof[iva]))
    return roc_auc_score(y, oof), oof, folds

def repeated_cv(X, y, label, seeds=(42, 2024, 7), model_fn=None):
    t0 = time.time(); s = [cv_once(X, y, sd, model_fn)[0] for sd in seeds]
    s = np.array(s)
    print(f"  {label:<32s} {s.mean():.5f} +/- {s.std(ddof=1):.5f} ({time.time()-t0:.0f}s)")
    return s

def fe_raw(df):
    X = df.drop(columns=[TARGET, "id"], errors="ignore").copy()
    for c in CATS: X[c] = X[c].astype("category")
    return X

Xr = fe_raw(train)
_, _, folds = cv_once(Xr, y, 42)
print("per-fold AUCs in ONE run:", " ".join(f"{f:.5f}" for f in folds))
print(f"per-fold range = {max(folds)-min(folds):.5f} <-- NOT the noise floor\n")
raw_scores = repeated_cv(Xr, y, "raw features, 3 partition seeds")
FLOOR = raw_scores.std(ddof=1)
print(f"\nNOISE FLOOR = {FLOOR:.5f} ({(max(folds)-min(folds))/FLOOR:.0f}x tighter "
      f"than the per-fold range)")


### セル3: データの構造とSimpsonのパラドックス（What / Why）

**何をしているか**: `daily_screen_time_hours = social_media_hours + gaming_hours + work_study_hours + その他` という会計的な恒等式が、このコンペのデータでは完全に成り立つ（違反ゼロ）ことを確認する。次に `work_study_hours`（勉強・仕事の画面時間）が単独では依存傾向と正の相関（AUC 0.6549、「害があるように見える」）を示す一方、`daily_screen_time_hours` を固定して層別に見ると、逆に依存率が下がる（Simpsonのパラドックス、交絡変数を無視すると相関の向きが逆転する現象）ことを確認する。

**なぜそうするのか**: 特徴量同士の関係を可視化せずにモデルに投げると、こうした構造（恒等式・交絡）を利用した特徴量エンジニアリングの機会を逃す。実際にこの後のセルで合成特徴量（比率・残差など）を作る際の裏付けとなる分析。またこのセルは「もっともらしい説明が実は間違っていた」という誠実な訂正（work_study_hoursが本当に保護的というより、単に合計を固定したことで算術的に情報を持ってしまっているだけ）も含んでおり、分析結果の解釈には注意が必要だという教訓を示している。


In [ ]:
ACC = ["daily_screen_time_hours", "social_media_hours", "gaming_hours", "work_study_hours"]
cp = train.dropna(subset=ACC)
resid = cp.daily_screen_time_hours - cp[ACC[1:]].sum(axis=1)
print(f"complete rows {len(cp):,} min residual {resid.min():.6f} "
      f"violations {(resid < -1e-9).sum():,}")

m = train.work_study_hours.notna()
print(f"\nwork_study_hours marginal AUC = "
      f"{roc_auc_score(y[m.values], train.loc[m,'work_study_hours']):.4f} (looks harmful)")
q = pd.qcut(train.daily_screen_time_hours, 5, labels=False, duplicates="drop")
qw = pd.qcut(train.work_study_hours, 5, labels=False, duplicates="drop")
grid = train.groupby([q, qw], observed=True)[TARGET].mean().unstack()
print("\naddiction rate, rows = daily_screen quintile, cols = work_study quintile:")
print(grid.round(3).to_string())


### セル4: 元データセット（現実のデータ）との比較（What / Why）

**何をしているか**: このPlaygroundコンペの合成データが生成元とした実データセット（Smartphone Usage and Addiction Prediction）を読み込み、同じ会計恒等式が現実データではどれくらい破れているか（60.7%の行で破れている）を確認する。さらに各特徴量の単独AUCを「現実データ」と「合成データ」で比較し、`work_study_hours` や `gaming_hours` が現実データではほぼ無意味（AUC≈0.50）なのに合成データでは予測力を持ってしまっている（生成過程が作り出した人工的な構造）ことを示す。

**なぜそうするのか**: Playgroundコンペのデータは実データを模した合成データであることが多く、生成器が意図せず作ってしまった統計的な構造（本来は無意味なはずの特徴量が予測に使えてしまう現象）を見つけることは、コンペのスコアを稼ぐ上で有効な診断になる。同時に「これは現実世界の因果関係ではなく、あくまでこの合成データセット特有の生成上の癖である」と明確に区別して書いている点が重要（セル3の「もっともらしいが誤った説明」を、実データとの比較によって検証・訂正している）。


In [ ]:
if ORIG_CSV is None:
    print("Attach 'jayjoshi37/smartphone-usage-and-addiction-prediction' to run this.")
else:
    orig = pd.read_csv(ORIG_CSV)
    r_o = orig.daily_screen_time_hours - orig[ACC[1:]].sum(axis=1)
    print(f"the accounting identity is violated in:")
    print(f"  synthetic (this competition): {(resid < -1e-9).mean():6.1%} of rows")
    print(f"  original (real data)        : {(r_o < -1e-9).mean():6.1%} of rows\n")
    cmp = pd.DataFrame([dict(feature=c,
                              real=roc_auc_score(orig.addicted_label, orig[c]),
                              synthetic=roc_auc_score(y[train[c].notna().values],
                                                        train.loc[train[c].notna(), c]))
                        for c in NUMS])
    cmp["gap"] = cmp.synthetic - cmp.real
    print(cmp.sort_values("gap", ascending=False).head(5)
          .to_string(index=False, float_format="%.4f"))


### セル5: 欠損値補完 — 「置き換える」のではなく「並べて追加する」（What / Why）

**何をしているか**: 各数値列について、他の列を特徴量としたXGBoost回帰モデルで欠損値を予測して埋める `impute` 関数を定義する（trainとtestを合わせて学習する「トランスダクティブ前処理」であり、目的変数を使っていないためリークではないと明記）。続いて特徴量エンジニアリング関数 `fe` で、比率・残差などの合成特徴量と欠損フラグを作る。補完した値で元の欠損列を**置き換える**方式と、元の欠損（NaN）列を**残したまま追加する**方式の2通りをクロスバリデーションで比較する。

**なぜそうするのか**: GBM（勾配ブースティング木）はノードごとに欠損値の分岐方向を学習でき、これは1つの補完点推定より表現力が高い。そのため補完値で「置き換える」と、この柔軟性を失ってかえって性能が下がる（実測 -0.00090）。一方、補完値を「追加」しつつ元のNaN列も残すと、モデルは「欠損しているという情報」と「補完された推定値」の両方を使えるため性能が上がる（実測 +0.00125）。同じ補完処理でも使い方次第で符号が逆転するという、実務上非常に重要な教訓を実験で示している。


In [ ]:
IMP = dict(n_estimators=400, learning_rate=0.08, max_depth=6, subsample=0.8,
           colsample_bytree=0.8, min_child_weight=20, tree_method="hist",
           device=DEVICE, enable_categorical=True)

def impute(tr_, te_, seed=42):
    """One XGB regressor per column, fit on train+test together. No target involved,
    so this is transductive preprocessing, not leakage."""
    n = len(tr_); full = pd.concat([tr_[NUMS+CATS], te_[NUMS+CATS]], ignore_index=True)
    X = full.copy()
    for c in CATS: X[c] = X[c].astype("category")
    out = full[NUMS].copy()
    for col in NUMS:
        obs = X[col].notna().values
        feats = [c for c in NUMS+CATS if c != col]
        m = xgb.XGBRegressor(**IMP, random_state=seed).fit(X.loc[obs, feats], X.loc[obs, col])
        if (~obs).sum(): out.loc[~obs, col] = m.predict(X.loc[~obs, feats])
    return out.iloc[:n].reset_index(drop=True), out.iloc[n:].reset_index(drop=True)

t0 = time.time(); tr_imp, te_imp = impute(train, test)
print(f"imputers fitted ({time.time()-t0:.0f}s)")

def fe(imp, orig, keep_raw):
    """Composition features on imputed values + missingness flags.
    keep_raw=True also keeps the ORIGINAL NaN-bearing columns alongside."""
    X = imp.copy()
    d, s, g = X.daily_screen_time_hours, X.social_media_hours, X.gaming_hours
    w, wk, sl = X.work_study_hours, X.weekend_screen_time, X.sleep_hours
    n, o = X.notifications_per_day, X.app_opens_per_day
    parts = s + g + w
    X["resid"], X["leisure"] = d - parts, d - w
    X["social_frac"], X["work_frac"] = s/d, w/d
    X["leisure_frac"], X["resid_frac"] = (d-w)/d, (d-parts)/d
    X["wk_ratio"], X["week_total"] = wk/d, 5*d + 2*wk
    X["awake_screen_frac"], X["free_time"] = d/(24-sl), 24 - sl - d - w
    X["notif_per_open"], X["min_per_open"] = n/o, d*60/o
    for c in CATS: X[c] = orig[c].astype("category").values
    for c in NUMS + CATS: X[f"na_{c}"] = orig[c].isna().astype(np.int8).values
    if keep_raw:
        for c in NUMS: X[f"rawnan_{c}"] = orig[c].values
    return X

X_replace = fe(tr_imp, train, False)
X_augment = fe(tr_imp, train, True)
print(f"replace: {X_replace.shape[1]} features augment: {X_augment.shape[1]} features\n")
rep = repeated_cv(X_replace, y, "imputed REPLACES the NaNs")
aug = repeated_cv(X_augment, y, "imputed ALONGSIDE the NaNs")
print(f"\nreplace: {rep.mean()-raw_scores.mean():+.5f} vs raw "
      f"({(rep.mean()-raw_scores.mean())/FLOOR:+.0f}x floor)")
print(f"augment: {aug.mean()-raw_scores.mean():+.5f} vs raw "
      f"({(aug.mean()-raw_scores.mean())/FLOOR:+.0f}x floor)")


### セル6: 最も効いた工夫 — 連続値を含む全カラムのターゲットエンコーディング（What / Why）

**何をしているか**: 数値・カテゴリを問わず全カラムを文字列レベルに変換し、各水準（level）ごとの「スムージングされたターゲット平均（target encoding）」と「出現頻度（frequency encoding）」を計算する準備をする。カラムごとの水準数と1水準あたりの行数を表示し、リーク対策として**ネストしたクロスバリデーション**（外側foldの学習データの中でさらに内側foldに分けてtarget encodingを計算する）の枠組み `build_enc` を定義する。

**なぜそうするのか**: 通常、連続値をカテゴリのように扱うのは危険（水準が多すぎて統計的信頼性が低い）とされるが、`daily_screen_time_hours` は1,389通りの値に対し1水準あたり約500行あり、スムージングされた平均が十分信頼できる推定量になる。木モデルは同じ曲線を近似するのに何十回も分岐が必要だが、target encodingなら1回の参照で済む。ネストしたクロスバリデーションを使うのは、target encodingが目的変数を直接使うため、素朴に計算するとCVスコアが不当に高くなり（リーク）、リーダーボードで崩壊するのを防ぐため。「自分の行の目的変数を、自分の行のエンコーディングが絶対に見ない」設計になっている。


In [ ]:
ENC_COLS = NUMS + CATS
print(f"{'column':<26s}{'levels':>8s}{'rows/level':>12s}")
for c in ENC_COLS:
    k = train[c].astype(str).nunique()
    print(f"{c:<26s}{k:>8,d}{len(train)/k:>12,.0f}")

SMOOTH = 10.0
LTR = pd.DataFrame({c: train[c].astype(str).values for c in ENC_COLS})
LTE = pd.DataFrame({c: test[c].astype(str).values for c in ENC_COLS})
ORDER = [f"te_{c}" for c in ENC_COLS] + [f"fq_{c}" for c in ENC_COLS]

def maps_from(levels, yy):
    gm = yy.mean(); m = {}
    for c in ENC_COLS:
        g = pd.DataFrame({"lv": levels[c].values, "y": yy}).groupby("lv")["y"].agg(["count","mean"])
        m[c] = (((g["count"]*g["mean"] + SMOOTH*gm)/(g["count"]+SMOOTH)).astype(np.float32),
                g["count"].astype(np.float32))
    return m, gm

def apply_maps(levels, m, gm):
    out = {}
    for c in ENC_COLS:
        tmap, fmap = m[c]
        out[f"te_{c}"] = levels[c].map(tmap).astype(np.float32).fillna(gm).values
        out[f"fq_{c}"] = levels[c].map(fmap).astype(np.float32).fillna(0.0).values
    return pd.DataFrame(out)[ORDER]

X_te_base = X_augment.reset_index(drop=True)
X_te_base_test = fe(te_imp, test, True).reset_index(drop=True)
FOLDS = list(StratifiedKFold(5, shuffle=True, random_state=42).split(X_te_base, y))

def build_enc(itr, iva):
    y_tr = y[itr]; L = LTR.iloc[itr].reset_index(drop=True)
    holder = np.zeros((len(itr), len(ORDER)), dtype=np.float32)
    for i_in, i_out in StratifiedKFold(5, shuffle=True, random_state=0).split(np.zeros(len(itr)), y_tr):
        m, gm = maps_from(L.iloc[i_in], y_tr[i_in])
        holder[i_out] = apply_maps(L.iloc[i_out].reset_index(drop=True), m, gm).values
    m, gm = maps_from(L, y_tr)
    return (pd.DataFrame(holder, columns=ORDER),
            apply_maps(LTR.iloc[iva].reset_index(drop=True), m, gm),
            apply_maps(LTE, m, gm))


### セル7: ターゲットエンコーディングの効果を実測（What / Why）

**何をしているか**: `run_te` 関数で、target encodingを使わない場合と使う場合のOOF（out-of-fold）AUCを実測して比較する。結果は0.96560 → 0.96801で、差は+0.00241（ノイズフロアの63倍）と、統計的に明確に有意な改善であることを確認する。

**なぜそうするのか**: 「効いたはず」という思い込みではなく、セル2で定義したノイズフロアと比較することで、この改善が測定誤差の範囲を大きく超える本物の効果であることを検証している。この後、実際のリーダーボードに提出する前に「CVからLBを予測できるか」を確認する誠実な検証手順（このノートブックの後半で言及）にもつながる。


In [ ]:
def run_te(use_enc, label, save=None):
    t0 = time.time(); oof = np.zeros(len(X_te_base)); tp = np.zeros(len(X_te_base_test))
    for f, (itr, iva) in enumerate(FOLDS):
        Xa = X_te_base.iloc[itr].reset_index(drop=True)
        Xb = X_te_base.iloc[iva].reset_index(drop=True)
        Xt = X_te_base_test
        if use_enc:
            e_tr, e_va, e_te = build_enc(itr, iva)
            Xa = pd.concat([Xa, e_tr], axis=1); Xb = pd.concat([Xb, e_va], axis=1)
            Xt = pd.concat([Xt, e_te], axis=1)
        m = mk(42); m.fit(Xa, y[itr], eval_set=[(Xb, y[iva])], verbose=False)
        oof[iva] = m.predict_proba(Xb)[:, 1]; tp += m.predict_proba(Xt)[:, 1] / 5
    a = roc_auc_score(y, oof)
    print(f"  {label:<34s} OOF {a:.5f} ({time.time()-t0:.0f}s)")
    if save: PRED[save] = (oof, tp)
    return a

PRED = {}
a_no = run_te(False, "no encodings (imputed+composition)", save="xgb_aug")
a_te = run_te(True, "+ target & frequency encodings", save="xgb_te")
print(f"\ndelta = {a_te - a_no:+.5f} ({(a_te-a_no)/FLOOR:+.0f}x noise floor)")


### セル8: 追加モデル（LightGBM・CatBoost）の学習（What / Why）

**何をしているか**: `member` 関数で、target encoding込みのLightGBM（`lgb_te`）と、target encodingなしのCatBoost（`cat_aug`、カテゴリ列はネイティブのcat_features機能に任せる）を追加で学習し、それぞれのOOF予測を `PRED` 辞書に蓄積する。

**なぜそうするのか**: この後のスタッキング（複数モデルの予測を組み合わせる）のために、異なる種類のモデル・異なる特徴量表現を意図的に用意している。特にCatBoostは他より弱い（AUC 0.9616）が、「弱いが他と相関が低い（decorrelated）」モデルとして、スタッキングにどう寄与するかを観察する目的でコメントされている。


In [ ]:
def member(name, kind, rep):
    t0 = time.time(); oof = np.zeros(len(X_te_base)); tp = np.zeros(len(X_te_base_test))
    for f, (itr, iva) in enumerate(FOLDS):
        Xa = X_te_base.iloc[itr].reset_index(drop=True)
        Xb = X_te_base.iloc[iva].reset_index(drop=True); Xt = X_te_base_test
        if rep == "te":
            e_tr, e_va, e_te = build_enc(itr, iva)
            Xa = pd.concat([Xa, e_tr], axis=1); Xb = pd.concat([Xb, e_va], axis=1)
            Xt = pd.concat([Xt, e_te], axis=1)
        if kind == "lgb":
            m = lgb.LGBMClassifier(n_estimators=4000, learning_rate=0.05, num_leaves=63,
                                    colsample_bytree=.8, subsample=.8, subsample_freq=1,
                                    min_child_samples=100, verbose=-1, random_state=42)
            m.fit(Xa, y[itr], eval_set=[(Xb, y[iva])], eval_metric="auc",
                  callbacks=[lgb.early_stopping(100, verbose=False)])
        else:
            Xa, Xb, Xt = [d.copy() for d in (Xa, Xb, Xt)]
            cc = [c for c in Xa.columns if isinstance(Xa[c].dtype, pd.CategoricalDtype)]
            for d in (Xa, Xb, Xt):
                for c in cc: d[c] = d[c].astype(object).fillna("__m__").astype(str)
            m = CatBoostClassifier(iterations=3000, learning_rate=0.06, depth=6,
                                    eval_metric="AUC", verbose=0, random_seed=42,
                                    early_stopping_rounds=100,
                                    **({"task_type": "GPU"} if DEVICE == "cuda" else {}))
            m.fit(Xa, y[itr], eval_set=(Xb, y[iva]),
                  cat_features=[Xa.columns.get_loc(c) for c in cc], verbose=0)
        oof[iva] = m.predict_proba(Xb)[:, 1]; tp += m.predict_proba(Xt)[:, 1] / 5
    PRED[name] = (oof, tp)
    print(f"  {name:<12s} {roc_auc_score(y, oof):.5f} ({time.time()-t0:.0f}s)")

member("lgb_te", "lgb", "te")
member("cat_aug", "cat", "aug")  # weak + different: watch what the stacker does with it
print()
for k, (o, _) in PRED.items(): print(f"  {k:<12s} {roc_auc_score(y, o):.5f}")


### セル9: スタッキング — ヒルクライミング法とロジスティック回帰の比較（What / Why）

**何をしているか**: 2種類の「複数モデルを組み合わせる方法（コンバイナ）」を比較する。1つは貪欲法で重みを加算していく「ヒルクライミング法」（重みは常に0以上）、もう1つは予測確率をロジット（対数オッズ）に変換した上でロジスティック回帰で重みを学習する「ロジットスタッキング」（重みが負にもなりうる）。それぞれをOOFの半分で学習しもう半分で評価する分割を5回繰り返し、ペアごとの差分（同じ行同士で比較するので分割ノイズが打ち消し合う）で有意性を確認する。

**なぜそうするのか**: ヒルクライミング法は加算的にしか重みを選べないため、弱いモデル（cat_aug）は完全に無視されてしまう（重み0）。一方、線形のロジスティック回帰スタッキングは負の係数を学習でき、弱いモデルの予測を「補正項」として活用できる（実際、cat_augの係数は負）。また確率のままではなくロジットスケールでスタッキングするのは、target（依存判定）が飽和領域（限りなく0または1に近い）を持ち、確率スケールでは分解能を失うが、ロジットスケールなら差を捉え続けられるため。単純な平均（mean_all）は最良の単体モデルより悪化するという結果も、「弱いモデルを機械的に平均に混ぜるのは危険」という教訓を示している。


In [ ]:
def to_logit(p, clip=30.0):
    p = np.clip(np.asarray(p, np.float64), 1e-15, 1-1e-15)
    return np.clip(np.log(p/(1-p)), -clip, clip)

def hill_climb(o, yy, n_iter=30):
    nm = list(o); P = np.column_stack([o[n] for n in nm]); picks = np.zeros(len(nm), int)
    j = int(np.argmax([roc_auc_score(yy, P[:, k]) for k in range(P.shape[1])]))
    picks[j] = 1; cur = P[:, j].copy(); best = roc_auc_score(yy, cur); k = 1
    for _ in range(n_iter):
        cb, cj = best, -1
        for t in range(P.shape[1]):
            a = roc_auc_score(yy, (cur*k + P[:, t])/(k+1))
            if a > cb + 1e-9: cb, cj = a, t
        if cj < 0: break
        cur = (cur*k + P[:, cj])/(k+1); k += 1; picks[cj] += 1; best = cb
    s = picks.sum() or 1
    return {n: v/s for n, v in zip(nm, picks)}

names = list(PRED)
Z = np.column_stack([to_logit(PRED[n][0]) for n in names])
R = np.column_stack([PRED[n][0] for n in names])
rows = []
for rep in range(5):
    iA, iB = next(StratifiedShuffleSplit(1, test_size=.5, random_state=rep)
                  .split(np.zeros(len(y)), y))
    w = hill_climb({n: PRED[n][0][iA] for n in names}, y[iA])
    rows.append(dict(
        best_solo=max(roc_auc_score(y[iB], PRED[n][0][iB]) for n in names),
        hill_climb=roc_auc_score(y[iB], sum(w[n]*PRED[n][0][iB] for n in names)),
        logit_stack=roc_auc_score(y[iB], LogisticRegression(max_iter=2000)
                                   .fit(Z[iA], y[iA]).predict_proba(Z[iB])[:, 1]),
        raw_stack=roc_auc_score(y[iB], LogisticRegression(max_iter=2000)
                                 .fit(R[iA], y[iA]).predict_proba(R[iB])[:, 1]),
        mean_all=roc_auc_score(y[iB], R[iB].mean(1))))
rob = pd.DataFrame(rows)
print(rob.to_string(float_format="%.6f"))
print("\nPAIRED DIFFERENCES (same rows, so split noise cancels)")
for a, b in [("logit_stack","hill_climb"), ("logit_stack","raw_stack"),
             ("logit_stack","best_solo"), ("mean_all","best_solo")]:
    d = rob[a] - rob[b]
    ok = "consistent" if (d > 0).all() or (d < 0).all() else "SIGN FLIPS"
    print(f"  {a:>12s} - {b:<12s} = {d.mean():+.6f} +/- {d.std(ddof=1):.6f} [{ok}]")

meta = LogisticRegression(max_iter=2000).fit(Z, y)
w_full = hill_climb({n: PRED[n][0] for n in names}, y)
print("\n" + pd.DataFrame({"solo": [roc_auc_score(y, PRED[n][0]) for n in names],
                            "hill_climb_w": [w_full[n] for n in names],
                            "stacker_coef": meta.coef_[0]}, index=names)
      .sort_values("stacker_coef", ascending=False).to_string(float_format="%.4f"))


### セル10: 最終提出の作成（What / Why）

**何をしているか**: テストデータに対する各モデルの予測をロジットスケールで合成し、最終的なロジスティック回帰スタッカーで確率に変換して `submission.csv` を書き出す。クロスフィットしたOOFスタックスコアも参考値として表示する。

**なぜそうするのか**: このノートブックは最後に「CVはリーダーボードスコアを直接推定するものではないが、CVの“差”は信頼できる」という重要な注意を述べている（実測: CV 0.966012→LB 0.96751、CV 0.966217→LB 0.96769。CVとLBの絶対値には常に約+0.0013のオフセットがあるが、2つのパイプライン間の差の順序はCVとLBで一致している）。つまりCVは「どちらが良いか」を判断する順位付けの道具として使うべきで、「本番でどのくらいのスコアになるか」を予言する道具として過信してはいけない、という結論。


In [ ]:
Zt = np.column_stack([to_logit(PRED[n][1]) for n in names])
pred = meta.predict_proba(Zt)[:, 1]
mo = np.zeros(len(y))
for itr, iva in StratifiedKFold(5, shuffle=True, random_state=42).split(Z, y):
    mo[iva] = LogisticRegression(max_iter=2000).fit(Z[itr], y[itr]).predict_proba(Z[iva])[:, 1]
print(f"cross-fitted stack OOF = {roc_auc_score(y, mo):.6f}")
sub = pd.DataFrame({"id": test["id"].values, TARGET: pred})
sub.to_csv("submission.csv", index=False)
print(f"submission.csv rows={len(sub):,} mean {pred.mean():.4f} "
      f"(train rate {y.mean():.4f})")
sub.head()
